In [1]:
import logging
from datetime import datetime

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

notebook_name = "Silver"
run_start = datetime.utcnow()
logger.info(f"[START] {notebook_name} — {run_start.strftime('%Y-%m-%d %H:%M:%S')} UTC")

StatementMeta(, b9b8c7e8-5a3e-4048-b119-76ba15bd72bc, 3, Finished, Available, Finished, False)

INFO:__main__:[START] Silver — 2026-04-24 03:07:20 UTC


In [2]:
# Run this at the top of each notebook to see what tables it touches
# Paste the output for all 4 notebooks

spark.sql("SHOW TABLES").show(50, truncate=False)

StatementMeta(, b9b8c7e8-5a3e-4048-b119-76ba15bd72bc, 4, Finished, Available, Finished, False)

+-----------------------------------------------------------------+-----------------------------+-----------+
|namespace                                                        |tableName                    |isTemporary|
+-----------------------------------------------------------------+-----------------------------+-----------+
|Vehicle_reliabillity_workspace.Vehicle_reliabillity_lakehouse.dbo|bronze_vehicle_images        |false      |
|Vehicle_reliabillity_workspace.Vehicle_reliabillity_lakehouse.dbo|complaints                   |false      |
|Vehicle_reliabillity_workspace.Vehicle_reliabillity_lakehouse.dbo|gold_brand_reliability       |false      |
|Vehicle_reliabillity_workspace.Vehicle_reliabillity_lakehouse.dbo|gold_component_failures      |false      |
|Vehicle_reliabillity_workspace.Vehicle_reliabillity_lakehouse.dbo|gold_dim_failure_category    |false      |
|Vehicle_reliabillity_workspace.Vehicle_reliabillity_lakehouse.dbo|gold_dim_make                |false      |
|Vehicle_r

In [3]:
from pyspark.sql.functions import (col, when, trim, upper,initcap,  lower, to_date, concat_ws,
    month, year, quarter, dayofweek,
    round as spark_round, lit, current_timestamp)


# read the delta table


df_silver_complaints = spark.read.format("delta").table("complaints")
df_silver_recalls = spark.read.format("delta").table("recalls")
df_silver_logos = spark.read.format("delta").table("logos")
df_silver_maintenance_data = spark.read.format("delta").table("maintenance_data")
df_silver_vehicle_images = spark.read.format("delta").table("vehicle_images")



print(f"Starting row count: {df_silver_complaints.count()}")
print(f"Starting row count: {df_silver_recalls.count()}")
print(f"Starting row count: {df_silver_logos.count()}")
print(f"Starting row count: {df_silver_maintenance_data.count()}")
print(f"Starting row count: {df_silver_vehicle_images.count()}")

StatementMeta(, b9b8c7e8-5a3e-4048-b119-76ba15bd72bc, 5, Finished, Available, Finished, False)

Starting row count: 178648
Starting row count: 8061
Starting row count: 27
Starting row count: 2662
Starting row count: 1102


In [4]:
# NULL Check

for c in df_silver_vehicle_images.columns:
    null_count = df_silver_vehicle_images.filter(col(c).isNull()).count()
    print(f"{c} has {null_count} null(s)")
    
    # Drop nulls for this column
    df_silver_vehicle_images = df_silver_vehicle_images.dropna(
    subset=["make", "url"]
)

StatementMeta(, b9b8c7e8-5a3e-4048-b119-76ba15bd72bc, 6, Finished, Available, Finished, False)

make has 552 null(s)
model has 0 null(s)
year has 0 null(s)
url has 0 null(s)
ingestion_timestamp has 0 null(s)
sourcefile has 0 null(s)
layer has 0 null(s)


In [5]:
# Standardizing  Complaints

df_silver_complaints = df_silver_complaints \
    .withColumn(
        "vehicle_id",
        concat_ws("_", col("vehicle_make"), col("vehicle_model"), col("model_year"))) \
    .withColumn("vehicle_make", initcap(col("vehicle_make"))) \
    .withColumn("vehicle_make",
        when(col("vehicle_make") == "Mercedes-Benz", "Mercedes-Benz")
        .when(col("vehicle_make") == "Mercedes-benz", "Mercedes-Benz")
        .otherwise(col("vehicle_make"))) \
    .withColumn("vehicle_model", initcap(col("vehicle_model"))) \
    .withColumn("component", initcap(col("component"))) \
    .withColumn("issue_description", lower(col("issue_description"))) \
    .withColumn("dataOfIncident",to_date(col("dateOfIncident"), "yyyy-mm-dd")) \
    .filter(col("vehicle_make").isNotNull()) \
    .withColumn("silver_timestamp", current_timestamp()) \
    .withColumn("layer", lit("silver"))





StatementMeta(, b9b8c7e8-5a3e-4048-b119-76ba15bd72bc, 7, Finished, Available, Finished, False)

In [6]:
# Standardizing  Recalls

df_silver_recalls = df_silver_recalls \
    .withColumn(
        "vehicle_id",
        concat_ws("_", col("vehicle_make"), col("vehicle_model"), col("model_year"))) \
    .withColumn("vehicle_make", initcap(col("vehicle_make"))) \
    .withColumn("vehicle_make",
        when(col("vehicle_make") == "Mercedes-Benz", "Mercedes-Benz")
        .when(col("vehicle_make") == "Mercedes-benz", "Mercedes-Benz")
        .otherwise(col("vehicle_make"))) \
    .withColumn("vehicle_model", initcap(col("vehicle_model"))) \
    .withColumn("component", initcap(col("component"))) \
    .withColumn("recall_issue", lower(col("recall_issue"))) \
    .withColumn("consequence", initcap(col("consequence"))) \
    .withColumn("recall_date",to_date(col("recall_date"), "yyyy-mm-dd")) \
    .filter(col("vehicle_make").isNotNull())



StatementMeta(, b9b8c7e8-5a3e-4048-b119-76ba15bd72bc, 8, Finished, Available, Finished, False)

In [7]:
from pyspark.sql.functions import when

# Apply to both df_silver_complaints and df_silver_recalls
df_silver_complaints = df_silver_complaints.withColumn(
    "vehicle_make",
    when(col("vehicle_make") == "Mercedes-Benz", "Mercedes-Benz")
    .when(col("vehicle_make") == "Mercedes-benz", "Mercedes-Benz")
    .otherwise(col("vehicle_make"))
)

df_silver_recalls = df_silver_recalls.withColumn(
    "vehicle_make",
    when(col("vehicle_make") == "Mercedes-Benz", "Mercedes-Benz")
    .when(col("vehicle_make") == "Mercedes-benz", "Mercedes-Benz")
    .otherwise(col("vehicle_make"))
)

StatementMeta(, b9b8c7e8-5a3e-4048-b119-76ba15bd72bc, 9, Finished, Available, Finished, False)

In [8]:
# Standardizing  Maintenance Data

df_silver_maintenance_data = df_silver_maintenance_data \
    .withColumnRenamed("make", "vehicle_make") \
    .withColumnRenamed("model", "vehicle_model") \
    .withColumnRenamed("year", "model_year") \
    .withColumn(
        "vehicle_id",
        concat_ws("_", col("vehicle_make"), col("vehicle_model"), col("model_year"))) \
    .withColumn("vehicle_make", initcap(col("vehicle_make"))) \
    .withColumn("vehicle_make",
        when(col("vehicle_make") == "Mercedes-Benz", "Mercedes-Benz")
        .when(col("vehicle_make") == "Mercedes-benz", "Mercedes-Benz")
        .otherwise(col("vehicle_make"))) \
    .withColumn("vehicle_model", initcap(col("vehicle_model"))) \
    .filter(col("vehicle_make").isNotNull())

StatementMeta(, b9b8c7e8-5a3e-4048-b119-76ba15bd72bc, 10, Finished, Available, Finished, False)

In [9]:
# Standardizing  Vehicle Images

df_silver_vehicle_images = df_silver_vehicle_images \
    .withColumnRenamed("make", "vehicle_make") \
    .withColumnRenamed("model", "vehicle_model") \
    .withColumnRenamed("year", "model_year") \
    .withColumn(
        "vehicle_id", 
        concat_ws("_", col("vehicle_make"), col("vehicle_model"), col("model_year"))
    )




StatementMeta(, b9b8c7e8-5a3e-4048-b119-76ba15bd72bc, 11, Finished, Available, Finished, False)

In [10]:
# Dropping columns not useful moving forward



columns_to_drop = ["sourcefile", "ingestion_timestamp"]
df_silver_complaints = df_silver_complaints.drop(*columns_to_drop)
print(f"Final Silver columns: {df_silver_complaints.columns}")



columns_to_drop = ["sourcefile", "ingestion_timestamp"]
df_silver_recalls = df_silver_recalls.drop(*columns_to_drop)
print(f"Final Silver columns: {df_silver_recalls.columns}")



columns_to_drop = ["sourcefile", "ingestion_timestamp"]
df_silver_logos = df_silver_logos.drop(*columns_to_drop)
print(f"Final Silver columns: {df_silver_logos.columns}")


columns_to_drop = ["sourcefile", "ingestion_timestamp"]
df_silver_maintenance_data = df_silver_maintenance_data.drop(*columns_to_drop)
print(f"Final Silver columns: {df_silver_logos.columns}")


columns_to_drop = ["sourcefile", "ingestion_timestamp"]
df_silver_vehicle_images = df_silver_vehicle_images.drop(*columns_to_drop)
print(f"Final Silver columns: {df_silver_logos.columns}")





StatementMeta(, b9b8c7e8-5a3e-4048-b119-76ba15bd72bc, 12, Finished, Available, Finished, False)

Final Silver columns: ['complaint_id', 'vehicle_make', 'vehicle_model', 'model_year', 'component', 'issue_description', 'dateOfIncident', 'layer', 'vehicle_id', 'dataOfIncident', 'silver_timestamp']
Final Silver columns: ['recall_id', 'vehicle_make', 'vehicle_model', 'model_year', 'recall_issue', 'recall_date', 'consequence', 'component', 'affected_units', 'layer', 'vehicle_id']
Final Silver columns: ['vehicle_make', 'logo_url', 'layer']
Final Silver columns: ['vehicle_make', 'logo_url', 'layer']
Final Silver columns: ['vehicle_make', 'logo_url', 'layer']


In [11]:
df_silver_maintenance_data.filter(col("vehicle_make").isin([
    "INFINITI", "JAGUAR", "VOLVO", "CADILLAC", "CHRYSLER",
    "LINCOLN", "ACURA", "BUICK", "DODGE", "GMC"
])).select("vehicle_make").distinct().show()

StatementMeta(, b9b8c7e8-5a3e-4048-b119-76ba15bd72bc, 13, Finished, Available, Finished, False)

+------------+
|vehicle_make|
+------------+
+------------+



In [12]:
# write to delta table

df_silver_complaints.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_complaints")
print("Complaimts Silver Delta table created successfully")    



# write to delta table

df_silver_recalls.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_recalls")
print("Recalls Silver Delta table created successfully")    


# write to delta table

df_silver_logos.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_logos")
print("Logos Silver Delta table created successfully")    


# write to delta table

df_silver_maintenance_data.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_maintenance_data")
print("Maintenance Data Silver Delta table created successfully")    


# write to delta table

df_silver_vehicle_images.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_vehicle_images")
print("Vehicle Images Silver Delta table created successfully")    




StatementMeta(, b9b8c7e8-5a3e-4048-b119-76ba15bd72bc, 14, Finished, Available, Finished, False)

Complaimts Silver Delta table created successfully
Recalls Silver Delta table created successfully
Logos Silver Delta table created successfully
Maintenance Data Silver Delta table created successfully
Vehicle Images Silver Delta table created successfully


In [13]:
# ── Pipeline run completion log ──────────────────────────────────
run_end = datetime.utcnow()
duration = (run_end - run_start).seconds
logger.info(f"[END] {notebook_name} — completed in {duration}s")

# Write run log to lakehouse for monitoring
from pyspark.sql import Row
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType

log_schema = StructType([
    StructField("notebook_name",  StringType(),   True),
    StructField("run_start",      TimestampType(), True),
    StructField("run_end",        TimestampType(), True),
    StructField("duration_secs",  IntegerType(),  True),
    StructField("status",         StringType(),   True),
])

log_row = [Row(
    notebook_name = notebook_name,
    run_start     = run_start,
    run_end       = run_end,
    duration_secs = duration,
    status        = "SUCCESS"
)]

log_df = spark.createDataFrame(log_row, schema=log_schema)

log_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("pipeline_run_log")

print(f"[DONE] {notebook_name} — {duration}s")

StatementMeta(, b9b8c7e8-5a3e-4048-b119-76ba15bd72bc, 15, Finished, Available, Finished, False)

INFO:__main__:[END] Silver — completed in 85s
INFO:trident_token_library_wrapper:Calling getAccessToken from PyTridentTokenLibrary


[DONE] Silver — 85s


In [14]:
spark.read.format("delta").load("Tables/dbo/pipeline_run_log").show(truncate=False)

StatementMeta(, b9b8c7e8-5a3e-4048-b119-76ba15bd72bc, 16, Finished, Available, Finished, False)

+----------------+--------------------------+--------------------------+-------------+-------+
|notebook_name   |run_start                 |run_end                   |duration_secs|status |
+----------------+--------------------------+--------------------------+-------------+-------+
|Bronze_ingestion|2026-04-24 02:32:21.73143 |2026-04-24 02:54:46.748086|1345         |SUCCESS|
|Silver          |2026-04-24 03:07:20.991761|2026-04-24 03:08:46.294357|85           |SUCCESS|
|Bronze          |2026-04-24 03:03:00.139947|2026-04-24 03:04:06.873314|66           |SUCCESS|
|NULL            |2026-04-23 16:56:04.262779|2026-04-23 17:22:49.896034|1605         |SUCCESS|
|NULL            |2026-04-23 19:06:23.265507|2026-04-23 19:26:40.750176|1217         |SUCCESS|
|NULL            |2026-04-23 17:36:41.728831|2026-04-23 17:39:29.644403|167          |SUCCESS|
|NULL            |2026-04-23 16:49:15.708794|2026-04-23 16:50:55.963152|100          |SUCCESS|
|NULL            |2026-04-23 17:28:59.379131|2026-